# CURE-Rec — run all remaining reviewer actions

Run this notebook top-to-bottom. No YAML configuration is changed. The CRN study uses an internal stochastic click-feedback variant because the archived baseline has zero click feedback and therefore cannot expose random-shock variance. Phase A is not rerun.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

CANDIDATES=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in CANDIDATES if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.config import load_settings
from cure_rec.pipeline import run_experiment
from cure_rec.revision_suite import paired_user_statistics

CONFIG=ROOT/'configs'/'curesim_full.yaml'
RESULTS=ROOT/'results'/'reviewer_phase_assets'
RUN_ALL=True
CRN_SEEDS=(300,301,302,303,304)
CRN_CLICK_FEEDBACK_WEIGHT=0.35
print('Root:',ROOT)
print('Registered YAML is read-only; CRN override:',CRN_CLICK_FEEDBACK_WEIGHT)

## 1. Inspect archived Phase B/C assets


In [ ]:
for name in ['phase_b_objective_constraint_ablation.csv','phase_c_sampled_shapley_fidelity.csv']:
    p=RESULTS/'tables'/name
    if p.exists():
        print('\n'+name)
        display(pd.read_csv(p))
    else: print('Missing:',p)

## 2. Simulator-backed CRN paired-difference study

Pairs are empty→repeat_cap (0→1) and repeat_cap→repeat_cap+tail_slot (1→5). The only CRN-specific change is the internal click-feedback variant; the registered YAML is not edited.

In [ ]:
def pair_value(settings, seed, a, b, common):
    cfg=settings.model_copy(deep=True)
    cfg.run.seed=int(seed)
    cfg.run.common_random_numbers=bool(common)
    cfg.simulator.click_feedback_weight=CRN_CLICK_FEEDBACK_WEIGHT
    cfg.run.name=f'crn-click-{common}-{seed}-{a}-{b}'
    cfg.run.output_root=ROOT/'runs'/'reviewer-crn-click'
    logger,game,_=run_experiment(cfg)
    diffs=[]
    for scenario in game.scenario_games.values():
        diffs.append(float(scenario.values[b].utility-scenario.values[a].utility))
    return float(np.mean(diffs)),logger.run_dir

def run_crn():
    settings=load_settings(CONFIG)
    rows=[]
    for a,b in ((0,1),(1,5)):
        for seed in CRN_SEEDS:
            crn,crn_dir=pair_value(settings,seed,a,b,True)
            iid,iid_dir=pair_value(settings,seed,a,b,False)
            rows.append({'seed':seed,'mask_a':a,'mask_b':b,'crn_difference':crn,'independent_difference':iid,'paired_difference':crn-iid,'crn_run':str(crn_dir),'independent_run':str(iid_dir)})
    frame=pd.DataFrame(rows)
    summary=frame.groupby(['mask_a','mask_b']).agg(crn_variance=('crn_difference','var'),independent_variance=('independent_difference','var'),crn_mean=('crn_difference','mean'),independent_mean=('independent_difference','mean'),n=('seed','count')).reset_index()
    summary['variance_ratio']=summary['crn_variance']/summary['independent_variance']
    out=RESULTS/'crn_click_feedback'
    out.mkdir(parents=True,exist_ok=True)
    frame.to_csv(out/'crn_paired_differences.csv',index=False)
    summary.to_csv(out/'crn_summary.csv',index=False)
    (out/'crn_manifest.json').write_text(json.dumps({'seeds':list(CRN_SEEDS),'pairs':[[0,1],[1,5]],'click_feedback_weight':CRN_CLICK_FEEDBACK_WEIGHT,'registered_config_changed':False,'claim_scope':'CURE-Sim stochastic click-feedback CRN diagnostic'},indent=2))
    return frame,summary

if RUN_ALL:
    crn_rows,crn_summary=run_crn()
    display(crn_summary)

## 3. External paired statistics (runs only if audited input exists)


In [ ]:
metrics_path=RESULTS/'per_user_metrics.csv'
if RUN_ALL and metrics_path.exists():
    metrics=pd.read_csv(metrics_path)
    required={'user_id','model','hit','ndcg'}
    missing=required-set(metrics.columns)
    if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
    ci,tests=paired_user_statistics(metrics)
    ci.to_csv(RESULTS/'paired_bootstrap_ci.csv',index=False)
    tests.to_csv(RESULTS/'paired_tests_holm.csv',index=False)
    display(ci);display(tests)
else:
    print('Phase D skipped: no audited per_user_metrics.csv')

## 4. Completion manifest


In [ ]:
manifest={'run_all':RUN_ALL,'phase_a':'archived_not_rerun','phase_bc':'archived_inspected','crn':'executed_click_feedback_variant','phase_d':'executed' if metrics_path.exists() else 'skipped_missing_per_user_metrics','scalability':'skipped_no_distinct_8_10_player_library','registered_config_changed':False,'crn_click_feedback_weight':CRN_CLICK_FEEDBACK_WEIGHT}
p=RESULTS/'run_all_remaining_manifest.json'
p.write_text(json.dumps(manifest,indent=2))
print(p)
print(json.dumps(manifest,indent=2))